# T08 — Eine Konversation führen

Eine Folgefrage wie "Who verified them?" (Wer hat sie verifiziert?) ergibt nur zusammen mit den vorherigen Runden Sinn. Dieses Notebook vergleicht zwei Wege, dem RAG-Agenten die Konversation zu übergeben, und zeigt das Umschreiben der Frage, das der Agent zusätzlich vornimmt.

| Mechanismus | Idee |
|---|---|
| **Vollständiger Verlauf** | Alle vorherigen Runden wörtlich mitschicken |
| **Zusammenfassung des Verlaufs** | Eine vom LLM geschriebene Zusammenfassung der vorherigen Runden mitschicken |
| **Umformulierung der Frage** | Die Frage vor dem Retrieval in eine eigenständige umschreiben (in `RAG` immer aktiv, sobald es einen Verlauf gibt) |

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Zuerst T01 ausführen: Dort entsteht der Vector Store unter `backend/db/data_vs.db`. Ist der Store leer, baut `load_or_build_vector_store()` ihn aus allen Dokumenten auf; das dauert einige Minuten.

In [ ]:
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles

from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)


def text_message(role: Roles, text: str) -> LLMMessage:
    return LLMMessage(role=role, content=[MessageContent(type="text", text=text)])

## Vollständiger Verlauf

Nach jeder Runde werden Frage und Antwort an `history` angehängt und mit der nächsten Frage mitgeschickt. Die gefundenen Chunks werden nicht gespeichert, sonst würde der Verlauf schnell wachsen. Trotzdem steigen Kosten und Kontextlänge mit jeder Runde.

Die dritte Frage lässt sich nur aus dem Verlauf beantworten.

In [ ]:
async def run_full_history(queries: list[str]) -> None:
    history: list[LLMMessage] = []
    for query in queries:
        print(f"\n>>> {query}")
        answer = await ask(agent, query, history)
        history += [text_message(Roles.USER, query), answer]


await run_full_history(
    [
        "Which pallets in our portfolio have a third-party verified EPD?",
        "Rewrite what you just said as a pirate, very concisely.",
        "Write minutes of our conversation so far.",
    ]
)

## Zusammenfassung des Verlaufs

Vor jeder Runde fasst ein LLM die vorherigen Runden kurz zusammen; die Zusammenfassung ersetzt den Verlauf. Der Kontext bleibt klein, aber Details aus früheren Runden können verloren gehen.

`SUMMARY_PROMPT` übersetzt:

> Fasse diese Konversation zwischen einer Nutzerin oder einem Nutzer und einem Assistenten in wenigen Sätzen zusammen. Beantworte keine Frage.

In [ ]:
SUMMARY_PROMPT = (
    "Summarize this conversation between a user and an assistant in a few "
    "sentences. Do not answer any question."
)


async def summarize(history: list[LLMMessage]) -> str:
    transcript = "\n".join(f"{m.role}: {m.content[0].text}" for m in history)
    response = await llm.generate(
        [
            text_message(Roles.SYSTEM, SUMMARY_PROMPT),
            text_message(Roles.USER, transcript),
        ]
    )
    return response.content[0].text or ""


async def run_with_summary(queries: list[str]) -> None:
    history: list[LLMMessage] = []
    for query in queries:
        print(f"\n>>> {query}")
        context = []
        if history:
            summary = await summarize(history)
            print(f"Summary: {summary}\n")
            context = [text_message(Roles.SYSTEM, f"Conversation so far: {summary}")]
        answer = await ask(agent, query, context)
        history += [text_message(Roles.USER, query), answer]


await run_with_summary(
    [
        "Which pallets in our portfolio have a third-party verified EPD?",
        "Rewrite what you just said as a pirate, very concisely.",
        "Write minutes of our conversation so far.",
    ]
)

## Umformulierung der Frage

"Who verified them?" findet für sich allein embedded nichts Brauchbares. Sobald es einen Verlauf gibt, lässt `RAG` das LLM die Frage zuerst in eine eigenständige umschreiben (`make_query_standalone` in `conversational_toolkit.utils.retriever`) und sucht mit der umgeschriebenen Frage. Das ist auch in beiden Durchläufen oben passiert.

Das DEBUG-Log dieses Moduls zeigt die ursprüngliche und die umformulierte Frage.

In [ ]:
sink = logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.utils.retriever",
    format="{message}",
)
await run_full_history(
    [
        "Which pallets in our portfolio have a third-party verified EPD?",
        "Who verified them?",
    ]
)
logger.remove(sink)